In [10]:
import sys
sys.path.append('/Users/eitanturok/good-vibrations/src3')

import json
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import colorsys
from sklearn.decomposition import PCA
from huggingface_hub import hf_hub_download
from vibrations_pipeline import _extract_signal, _normalize_fft

REPO = 'eturok-weizmann/laser-vibrations'
SPEAKERS = ['0001', '0010', '0100', '1000']

## Select 9 positions with all 4 speakers

Load only the lightweight `manifest.json` files (no FFT data yet) to build a position→speaker index, then pick 9 positions spread across the dataset.

In [2]:
# # Uncomment to re-derive sample IDs from the repo (slow — downloads all 551 manifests)
# all_files = list(list_repo_files(REPO, repo_type='dataset'))
# all_sample_ids = sorted(
#     s for s in set(f.split('/')[1] for f in all_files if f.startswith('data/'))
#     if s != 'metadata.jsonl'
# )
# all_meta = []
# for sid in all_sample_ids:
#     path = hf_hub_download(REPO, f'data/{sid}/manifest.json', repo_type='dataset')
#     with open(path) as f:
#         m = json.load(f)
#     all_meta.append({
#         'sample_id': sid,
#         'speakers':  m['sample']['speakers'],
#         'x_com':     round(m['segmentation']['x_com']),
#         'y_com':     round(m['segmentation']['y_com']),
#     })
# by_pos = defaultdict(list)
# for r in all_meta:
#     by_pos[(r['x_com'], r['y_com'])].append(r)
# complete = {
#     pos: {r['speakers']: r['sample_id'] for r in items}
#     for pos, items in by_pos.items()
#     if set(r['speakers'] for r in items) >= set(SPEAKERS)
# }
# sorted_positions = sorted(complete.keys())
# idx9 = np.linspace(0, len(sorted_positions) - 1, 9, dtype=int)
# chosen_positions = [sorted_positions[i] for i in idx9]
# sample_ids = [complete[pos][spk] for pos in chosen_positions for spk in SPEAKERS]

In [8]:
# 9 positions × 4 speakers = 36 samples, evenly spread across 119 complete positions.
# Each group of 4 consecutive IDs shares the same (x_com, y_com); order is 0001,0010,0100,1000.
chosen_positions = [
    (193, 470), (271, 413), (346, 468), (460, 416), (565, 534),
    (643, 136), (790, 416), (929, 537), (984, 480),
]
sample_ids = [
    '0000517', '0000518', '0000519', '0000084',   # pos (193, 470)
    '0000213', '0000214', '0000215', '0000216',   # pos (271, 413)
    '0000261', '0000262', '0000263', '0000264',   # pos (346, 468)
    '0000541', '0000542', '0000539', '0000540',   # pos (460, 416)
    '0000321', '0000322', '0000323', '0000324',   # pos (565, 534)
    '0000021', '0000022', '0000023', '0000024',   # pos (643, 136)
    '0000241', '0000242', '0000243', '0000244',   # pos (790, 416)
    '0000341', '0000342', '0000343', '0000344',   # pos (929, 537)
    '0000297', '0000298', '0000299', '0000300',   # pos (984, 480)
]
print(f'Selected {len(sample_ids)} samples (9 positions × 4 speakers)')

Selected 36 samples (9 positions × 4 speakers)


## Load manifest + FFT for each sample

Only downloading `manifest.json` and `speckle_shifts_fft.npz` — skip video, audio, raw npy.

In [16]:
def load_sample(sample_id):
    manifest_path = hf_hub_download(REPO, f'data/{sample_id}/manifest.json', repo_type='dataset')
    fft_path = hf_hub_download(REPO, f'data/{sample_id}/speckle_shifts_fft.npz', repo_type='dataset')

    with open(manifest_path) as f:
        manifest = json.load(f)

    fft_data = np.load(fft_path)
    fft   = fft_data['fft']    # (L, F, C) complex64
    freqs = fft_data['freqs']  # (F,)

    X_raw  = _extract_signal(fft, 'magnitude')               # (L, F, C)
    X_norm = _normalize_fft(X_raw[None], 'std-sample').squeeze(0)  # (L, F, C)

    return dict(
        sample_id = sample_id,
        X_raw     = X_raw,
        X_norm    = X_norm,
        freqs     = freqs,
        x_com     = manifest['segmentation']['x_com'],
        y_com     = manifest['segmentation']['y_com'],
        speakers  = manifest['sample']['speakers'],
    )

samples = [load_sample(sid) for sid in sample_ids]
for s in samples:
    print(f"{s['sample_id']}  speakers={s['speakers']}  com=({s['x_com']:.0f},{s['y_com']:.0f})"
          f"  X_raw={s['X_raw'].shape}  X_norm={s['X_norm'].shape}")

0000517  speakers=0001  com=(193,470)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000518  speakers=0010  com=(193,470)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000519  speakers=0100  com=(193,470)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000084  speakers=1000  com=(193,470)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000213  speakers=0001  com=(271,413)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000214  speakers=0010  com=(271,413)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000215  speakers=0100  com=(271,413)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000216  speakers=1000  com=(271,413)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000261  speakers=0001  com=(346,468)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000262  speakers=0010  com=(346,468)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000263  speakers=0100  com=(346,468)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000264  speakers=1000  com=(346,468)  X_raw=(100, 3421, 2)  X_norm=(100, 3421, 2)
0000

## Build feature vectors

Use FFT magnitude averaged over the 100 lasers and both XY channels → shape `(F,)` per sample.

In [17]:
# Verify all samples share the same frequency axis (allow tiny float differences)
ref_freqs = samples[0]['freqs']
assert all(np.allclose(s['freqs'], ref_freqs, atol=1e-3) for s in samples), "Frequency axes differ!"

# Average X_norm over lasers (axis 0) and xy channels (axis 2) → (F,) per sample,
# exactly as notebook 32 does: X_norm.mean(axis=(0, 2))
X = np.stack([s['X_norm'].mean(axis=(0, 2)) for s in samples])  # (36, F)

speakers_list  = [s['speakers'] for s in samples]
pos_labels     = [f"({s['x_com']:.0f}, {s['y_com']:.0f})" for s in samples]

print(f'Feature matrix: {X.shape}  (36 samples = 9 positions × 4 speakers)')
print(f'Unique speakers: {sorted(set(speakers_list))}')
print(f'Unique positions: {sorted(set(pos_labels))}')

Feature matrix: (36, 3421)  (36 samples = 9 positions × 4 speakers)
Unique speakers: ['0001', '0010', '0100', '1000']
Unique positions: ['(193, 470)', '(271, 413)', '(346, 468)', '(460, 416)', '(565, 534)', '(643, 136)', '(790, 416)', '(929, 537)', '(984, 480)']


## PCA

In [18]:
pca = PCA(n_components=3)
pcs = pca.fit_transform(X)
print(f'PCA explained variance ratio: {pca.explained_variance_ratio_}')
print(f'PC coordinates shape: {pcs.shape}')

PCA explained variance ratio: [0.65449371 0.15344961 0.08273061]
PC coordinates shape: (36, 3)


## 3D scatter: color = speaker, marker = position

In [20]:
def hsl_to_hex(h, s, l):
    r, g, b = colorsys.hls_to_rgb(h / 360, l / 100, s / 100)
    return f'#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}'

unique_speakers   = sorted(set(speakers_list))
unique_pos_labels = sorted(set(pos_labels))

# Plotly 3D only has 8 valid marker symbols — cycle through them and vary size for 9th+
PLOTLY_MARKERS = ['circle', 'square', 'diamond', 'cross', 'x',
                  'circle-open', 'square-open', 'diamond-open']
PLOTLY_SIZES   = [10, 16]  # alternate size when symbols repeat

spk_to_color  = {s: hsl_to_hex(int(360 * i / len(unique_speakers)), 70, 50)
                 for i, s in enumerate(unique_speakers)}
pos_to_marker = {p: PLOTLY_MARKERS[i % len(PLOTLY_MARKERS)] for i, p in enumerate(unique_pos_labels)}
pos_to_size   = {p: PLOTLY_SIZES[i // len(PLOTLY_MARKERS)] for i, p in enumerate(unique_pos_labels)}

traces = {}
for pt, spk, pos_lbl, s in zip(pcs, speakers_list, pos_labels, samples):
    key = (spk, pos_lbl)
    if key not in traces:
        traces[key] = dict(x=[], y=[], z=[], text=[], speaker=spk, pos_lbl=pos_lbl,
                           marker_symbol=pos_to_marker[pos_lbl],
                           marker_size=pos_to_size[pos_lbl],
                           color=spk_to_color[spk])
    traces[key]['x'].append(pt[0])
    traces[key]['y'].append(pt[1])
    traces[key]['z'].append(pt[2])
    traces[key]['text'].append(
        f"sample {s['sample_id']}<br>speakers {spk}<br>COM {pos_lbl}"
    )

data_traces = [
    go.Scatter3d(x=t['x'], y=t['y'], z=t['z'], text=t['text'], mode='markers',
                 hovertemplate='%{text}<extra></extra>', showlegend=False,
                 marker=dict(size=t['marker_size'], symbol=t['marker_symbol'], color=t['color'],
                             line=dict(width=1, color='black')))
    for t in traces.values()
]

# Legend: speaker (color)
spk_legend = [
    go.Scatter3d(x=[None], y=[None], z=[None], mode='markers',
                 name=f'spk {s}', legendgroup=f'spk_{i}',
                 legendgrouptitle=dict(text='Speaker') if i == 0 else {},
                 marker=dict(size=8, symbol='circle', color=spk_to_color[s]))
    for i, s in enumerate(unique_speakers)
]

# Legend: position (marker shape + size)
pos_legend = [
    go.Scatter3d(x=[None], y=[None], z=[None], mode='markers',
                 name=p, legendgroup=f'pos_{i}',
                 legendgrouptitle=dict(text='Position (x_com, y_com)') if i == 0 else {},
                 marker=dict(size=pos_to_size[p], symbol=pos_to_marker[p], color='grey',
                             line=dict(width=1, color='black')))
    for i, p in enumerate(unique_pos_labels)
]

fig = go.Figure(data_traces + spk_legend + pos_legend)
evr = pca.explained_variance_ratio_
fig.update_layout(
    title='PCA of FFT magnitudes (9 positions × 4 speakers) — color=speaker, shape=position',
    width=1400, height=800,
    scene=dict(
        xaxis_title=f'PC1 ({evr[0]*100:.1f}%)',
        yaxis_title=f'PC2 ({evr[1]*100:.1f}%)',
        zaxis_title=f'PC3 ({evr[2]*100:.1f}%)',
    ),
    legend=dict(groupclick='togglegroup')
)
fig.show()

## 2D PCA: PC1 vs PC2, colored by speaker and by position

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

pos_to_idx = {p: i for i, p in enumerate(unique_pos_labels)}
spk_to_idx = {s: i for i, s in enumerate(unique_speakers)}
mpl_markers = ['o', 's', 'D', '^', 'P', 'X', 'v', '<', '>']  # 9 shapes for 9 positions
evr = pca.explained_variance_ratio_

# --- Left: color=speaker, marker=position ---
ax = axes[0]
for pt, spk, pos_lbl in zip(pcs, speakers_list, pos_labels):
    color = hsl_to_hex(int(360 * spk_to_idx[spk] / len(unique_speakers)), 70, 50)
    ax.scatter(pt[0], pt[1], color=color,
               marker=mpl_markers[pos_to_idx[pos_lbl]], s=120,
               zorder=3, linewidths=0.5, edgecolors='k')

spk_handles = [
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor=hsl_to_hex(int(360 * i / len(unique_speakers)), 70, 50),
               markersize=9, label=f'spk {s}', markeredgecolor='k', markeredgewidth=0.5)
    for i, s in enumerate(unique_speakers)
]
pos_handles = [
    plt.Line2D([0],[0], marker=mpl_markers[i], color='w', markerfacecolor='grey',
               markersize=9, label=p, markeredgecolor='k', markeredgewidth=0.5)
    for i, p in enumerate(unique_pos_labels)
]
ax.legend(handles=spk_handles + pos_handles, title='Color=speaker  Shape=position',
          fontsize=7, ncol=2)
ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
ax.set_title('Do same-speaker points (same color) cluster?')
ax.grid(True, alpha=0.3)

# --- Right: color=position, marker=speaker (for reference) ---
ax = axes[1]
cmap9 = plt.cm.tab10
for pt, spk, pos_lbl in zip(pcs, speakers_list, pos_labels):
    color = cmap9(pos_to_idx[pos_lbl] / max(1, len(unique_pos_labels) - 1))
    ax.scatter(pt[0], pt[1], color=color,
               marker=mpl_markers[spk_to_idx[spk]], s=120,
               zorder=3, linewidths=0.5, edgecolors='k')

pos_handles2 = [
    plt.Line2D([0],[0], marker='o', color='w',
               markerfacecolor=cmap9(i / max(1, len(unique_pos_labels) - 1)),
               markersize=9, label=p, markeredgecolor='k', markeredgewidth=0.5)
    for i, p in enumerate(unique_pos_labels)
]
spk_handles2 = [
    plt.Line2D([0],[0], marker=mpl_markers[i], color='w', markerfacecolor='grey',
               markersize=9, label=f'spk {s}', markeredgecolor='k', markeredgewidth=0.5)
    for i, s in enumerate(unique_speakers)
]
ax.legend(handles=pos_handles2 + spk_handles2, title='Color=position  Shape=speaker',
          fontsize=7, ncol=2)
ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)')
ax.set_title('Do same-position points (same color) cluster?')
ax.grid(True, alpha=0.3)

plt.suptitle('PCA of FFT magnitudes — 9 positions × 4 speakers', fontsize=13)
plt.tight_layout()
plt.show()

## PCA loadings: which frequencies drive PC1 and PC2?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
for i, ax in enumerate(axes):
    ax.plot(ref_freqs, pca.components_[i])
    ax.set_ylabel(f'PC{i+1} loading')
    ax.set_title(f'PC{i+1} ({evr[i]*100:.1f}% variance)')
    ax.axhline(0, color='k', linewidth=0.5)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel('Frequency (Hz)')
plt.suptitle('PCA loadings — which frequencies matter')
plt.tight_layout()
plt.show()